In [ ]:
# =========================
# 0) Install + Login W&B
# =========================
!pip -q install wandb scikit-learn imbalanced-learn kagglehub

import wandb
wandb.login()

import time
PROJECT = "fl-fraud-compare"
GROUP = "compare4-" + str(int(time.time()))  # COPY this GROUP to other notebooks for comparison
print("W&B GROUP =", GROUP)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e20285 (e20285-university-of-peradeniya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B GROUP = compare4-1766841017


In [ ]:
run = wandb.init(
    project=PROJECT,
    group=GROUP,
    job_type="preprocess+train",
    name="Approach1-MLP",  # change per notebook
    config={
        "approach": "Approach1_MLP",   # change per notebook
        "model": "MLP",
        "dp": False,
        "num_clients": 10,
        "num_rounds": 5,
        "local_epochs": 1,
        "lr": 1e-3,
        "batch_size": 32,
        "fedavg": True,
        "smote": True,
        "scaler": "StandardScaler",
        "threshold": 0.5
    }
)

In [ ]:
# =========================
# 1) Load + preprocess data
# =========================
import kagglehub
import pandas as pd
import numpy as np

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


In [ ]:
# KaggleHub returns a local path; the CSV is inside that folder
# Let's find it robustly:
import os, glob
csv_candidates = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
assert len(csv_candidates) > 0, f"No CSV found under {path}"
file_path = csv_candidates[0]
print("Using CSV:", file_path)

Using CSV: /kaggle/input/creditcardfraud/creditcard.csv


In [ ]:
df = pd.read_csv(file_path)

In [ ]:
# Basic cleaning
df = df.drop_duplicates()
df.fillna(df.mean(numeric_only=True), inplace=True)

In [ ]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
if run.config["smote"]:
    smote = SMOTE(sampling_strategy="minority", random_state=42)
    X_res, y_res = smote.fit_resample(X_scaled, y)
else:
    X_res, y_res = X_scaled, y

In [ ]:
# split
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

In [ ]:
# log dataset stats to UI
run.summary["data/n_rows_total"] = int(len(df))
run.summary["data/n_features"] = int(X_train.shape[1])
run.summary["data/train_size"] = int(len(X_train))
run.summary["data/test_size"] = int(len(X_test))
run.summary["data/fraud_rate_after_resample"] = float(np.mean(y_res))

In [ ]:
# =========================
# 2) Split across FL clients
# =========================
num_clients = run.config["num_clients"]
client_data = np.array_split(X_train, num_clients)
client_labels = np.array_split(y_train, num_clients)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
# =========================
# 3) Define model + FL helpers
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))  # outputs probability
        return x


In [ ]:
def federated_averaging(models, device=device):
    global_model = MLP(in_dim=X_train.shape[1]).to(device)
    avg_weights = [torch.zeros_like(param.data) for param in global_model.parameters()]

    for m in models:
        for idx, param in enumerate(m.parameters()):
            avg_weights[idx] += param.data.to(device) / len(models)

    with torch.no_grad():
        for idx, param in enumerate(global_model.parameters()):
            param.data.copy_(avg_weights[idx])

    return global_model

In [ ]:
def local_train(client_x, client_y, model, epochs=1, lr=1e-3, batch_size=32):
    model.train()
    model.to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    inputs = torch.tensor(client_x, dtype=torch.float32)
    targets = torch.tensor(
        client_y.values if hasattr(client_y, "values") else client_y,
        dtype=torch.float32
    ).view(-1, 1)

    dataset = TensorDataset(inputs, targets)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for _ in range(epochs):
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            probs = model(bx)
            loss = criterion(probs, by)
            loss.backward()
            optimizer.step()

    return model


In [ ]:
@torch.no_grad()
def evaluate_global(model, X_eval, y_eval):
    model.eval()
    model.to(device)
    criterion = nn.BCELoss()

    X_t = torch.tensor(X_eval, dtype=torch.float32).to(device)
    y_t = torch.tensor(
        y_eval.values if hasattr(y_eval, "values") else y_eval,
        dtype=torch.float32
    ).view(-1, 1).to(device)

    probs = model(X_t)  # (N,1)
    loss = criterion(probs, y_t).item()

    preds = (probs >= run.config["threshold"]).float()
    acc = preds.eq(y_t).float().mean().item()

    return loss, acc, probs.detach().cpu().numpy().reshape(-1), y_t.detach().cpu().numpy().reshape(-1)


In [ ]:
# =========================
# 4) Metrics for W&B UI
# =========================
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support
)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = float((y_pred == y_true).mean())

    # AUCs may fail if only one class present
    roc_auc = None
    pr_auc = None
    if len(np.unique(y_true)) > 1:
        roc_auc = float(roc_auc_score(y_true, y_prob))
        pr_auc = float(average_precision_score(y_true, y_prob))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    return {
        "acc": acc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "y_pred": y_pred
    }


In [ ]:
# =========================
# 5) Federated training + W&B logging
# =========================
num_rounds = run.config["num_rounds"]
local_epochs = run.config["local_epochs"]
lr = run.config["lr"]
batch_size = run.config["batch_size"]

models = [MLP(in_dim=X_train.shape[1]) for _ in range(num_clients)]

for r in range(num_rounds):
    print(f"\n=== Round {r+1}/{num_rounds} ===")

    # Local training
    for i in range(num_clients):
        models[i] = local_train(
            client_data[i],
            client_labels[i],
            models[i],
            epochs=local_epochs,
            lr=lr,
            batch_size=batch_size
        )

    # Aggregate global model
    global_model = federated_averaging(models)

    # (Optional) push global weights back to clients
    for i in range(num_clients):
        models[i].load_state_dict(global_model.state_dict())

    # Evaluate global per round
    g_loss, g_acc, g_probs, g_true = evaluate_global(global_model, X_test, y_test)
    m = compute_metrics(g_true, g_probs, threshold=run.config["threshold"])

    log_dict = {
        "round": r + 1,
        "global/loss": g_loss,
        "global/acc": m["acc"],
        "global/precision": m["precision"],
        "global/recall": m["recall"],
        "global/f1": m["f1"],
    }
    if m["roc_auc"] is not None:
        log_dict["global/auc"] = m["roc_auc"]
    if m["pr_auc"] is not None:
        log_dict["global/pr_auc"] = m["pr_auc"]

    wandb.log(log_dict, step=r)

    print(
        f"Global eval — loss: {g_loss:.4f}, acc: {m['acc']:.4f}, "
        f"f1: {m['f1']:.4f}, auc: {m['roc_auc']}"
    )


=== Round 1/5 ===
Global eval — loss: 1.2900, acc: 0.5000, f1: 0.0000, auc: 0.22682041632202513

=== Round 2/5 ===
Global eval — loss: 0.0260, acc: 0.9942, f1: 0.9943, auc: 0.9996670339078675

=== Round 3/5 ===
Global eval — loss: 0.0194, acc: 0.9973, f1: 0.9973, auc: 0.9997749383221991

=== Round 4/5 ===
Global eval — loss: 0.0183, acc: 0.9976, f1: 0.9976, auc: 0.9997876068313614

=== Round 5/5 ===
Global eval — loss: 0.0172, acc: 0.9981, f1: 0.9981, auc: 0.9997898118047386


In [ ]:
# =========================
# 6) Final UI panels (CM + ROC + PR) + summaries
# =========================
# final metrics in summary (sortable)
run.summary["final/test_loss"] = float(g_loss)
run.summary["final/test_acc"] = float(m["acc"])
run.summary["final/test_precision"] = float(m["precision"])
run.summary["final/test_recall"] = float(m["recall"])
run.summary["final/test_f1"] = float(m["f1"])
if m["roc_auc"] is not None:
    run.summary["final/test_auc"] = float(m["roc_auc"])
if m["pr_auc"] is not None:
    run.summary["final/test_pr_auc"] = float(m["pr_auc"])

# confusion matrix plot
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=np.asarray(g_true).astype(int),
        preds=m["y_pred"],
        class_names=["legit", "fraud"]
    )
})

In [ ]:
# ROC + PR need Nx2 probs
probas_2col = np.stack([1 - g_probs, g_probs], axis=1)

wandb.log({
    "roc": wandb.plot.roc_curve(
        np.asarray(g_true).astype(int),
        probas_2col,
        labels=["legit", "fraud"]
    )
})

# PR curve (try; if unavailable, it will print the error)
try:
    wandb.log({
        "pr": wandb.plot.pr_curve(
            np.asarray(g_true).astype(int),
            probas_2col,
            labels=["legit", "fraud"]
        )
    })
except Exception as e:
    print("PR curve plot not available in this wandb version:", e)

print("\nFederated training complete.")
run.finish()


Federated training complete.


global/acc,▁████
global/auc,▁████
global/f1,▁████
global/loss,█▁▁▁▁
global/pr_auc,▁████
global/precision,▁████
global/recall,▁████
round,▁▃▅▆█
data/fraud_rate_after_resample,0.5
data/n_features,30
data/n_rows_total,283726
